# 04 · Pseudobulk per cell type × sample (Kolla et al. 2020, E16)

Notebook 03 built metacells and swept how to size them. Its result was that the sizing knobs barely
move score accuracy, while the **statistical** unit is what actually constrains the analysis:
metacells within one sample are partitions of the same cells from the same animal, not replicates.
Treating them as independent is pseudoreplication.

So this notebook builds the unit the replication structure can actually support — **one pseudobulk
profile per cell type × sample**, 18 × 3 = 54 units, n = 3 per cell type, paired across samples.
That is simultaneously the deepest and least sparse unit available, and it gives every per-gene call
a built-in reproducibility test.

**What changes from 03**

- No size target and no cap: the unit is the whole stratum, so there is nothing left to pool.
- Gene classes come from **replication rather than prediction** — a gene is `pooled` if all three
  samples detect it, `partial` if they disagree, `off` if none detect it and the cell type is deep
  enough that ambient-level expression would have shown, `uncertain` if none detect it and it is too
  shallow to tell absence from dropout.
- `off` uses the ambient floor from notebook 03's work. **Check the per-control table in Part 2
  before trusting it** — on this dataset Sdsl turned out to be expressed in cochlea and had to be
  dropped as a control.

**What this does not do.** Flux sampling and the archetype clustering come later, and this notebook
only prepares their input. It also forecloses discovering *within*-cell-type metabolic heterogeneity;
notebook 02 looked for that in E16 and found none, but it looked at dropout patterns rather than
flux, so treat that as suggestive rather than settled.

## 0 · Settings

In [ ]:
import os, sys, json

BASE = '/scratch/prj/crb_inner_ear/k2147692/metabolic'
REPO_DIR = f'{BASE}/code/Metabolic-pipeline'
DATA_PATH = f'{BASE}/data/kolla/kolla_E16.h5ad'
OUT_DIR = f'{BASE}/results/04_pseudobulk_E16'
MC_DIR = f'{BASE}/results/03_metabolic_metacells_E16'   # for the comparison in Part 4

CELLTYPE_COL = 'cell_type'
SAMPLE_COL = 'sample'
SYMBOL_COL = 'gene_symbol'
SPECIES = 'mmusculus'
AND_STRATEGY = 'median'      # how reaction scores are read
OR_STRATEGY = 'sum'
SPLIT_ISOZYMES = False       # weight an isozyme by its share of the summed reaction (flux bounds)
DETECTION_PROB = 0.95
NULL_MODEL = 'nb'
CONFIDENCE = 0.95            # confidence for the "would have seen it" test behind 'off'
FLOOR_QUANTILE = 0.95

sys.path.insert(0, REPO_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from metabolic_tools.metacell_diagnostics import recover_counts, gene_dropout_leverage, dropout_diagnostic
from metabolic_tools.metabolic_metacells import (
    aggregate_metacells, ambient_floor, pseudobulk_gene_classes, classify_reactions,
    AMBIENT_CONTROL_SYMBOLS)
from metabolic_tools.gene_mapping import resolve_model_path

adata = sc.read_h5ad(DATA_PATH)
adata.obs[SAMPLE_COL] = adata.obs_names.str.split('_E16').str[0]
report = recover_counts(adata)
print('Count recovery integer fraction:', report['integer_fraction'])
print(adata.obs.groupby([CELLTYPE_COL, SAMPLE_COL], observed=True).size().unstack(fill_value=0).to_string())

## Part 1 · Build the pseudobulk units
One profile per cell type × sample. Counts are summed and then renormalised to log1p(CP10K), the
same scale `calculate_ecs` expects — summing counts rather than averaging normalised values is what
makes the pooled profile a genuinely deeper measurement, and weights each cell by the evidence it
carries instead of equally.

In [ ]:
labels = (adata.obs[CELLTYPE_COL].astype(str) + '|' + adata.obs[SAMPLE_COL].astype(str))
pb = aggregate_metacells(adata, labels, CELLTYPE_COL, SAMPLE_COL)
print(f'{adata.n_obs} cells -> {pb.n_obs} pseudobulk units '
      f'({pb.obs[CELLTYPE_COL].nunique()} cell types x {pb.obs[SAMPLE_COL].nunique()} samples)')
print('Units mixing cell types:', int((pb.obs['celltype_purity'] < 1).sum()),
      '| mixing samples:', int((pb.obs['sample_purity'] < 1).sum()))

depth = (pb.obs.pivot_table(index=CELLTYPE_COL, columns=SAMPLE_COL, values='total_umis', observed=True)
         .assign(total=lambda d: d.sum(axis=1)).sort_values('total', ascending=False))
display(depth.round(0))

In [ ]:
# How much deeper is a pseudobulk unit than the metacells notebook 03 built?
try:
    mc_targets = pd.read_csv(os.path.join(MC_DIR, 'size_targets.csv')).set_index('group')
    cmp = pb.obs.groupby(CELLTYPE_COL, observed=True)['total_umis'].median().rename('pseudobulk_umis').to_frame()
    cmp['metacell_budget'] = mc_targets['target_umis']
    cmp['ratio'] = (cmp['pseudobulk_umis'] / cmp['metacell_budget']).round(2)
    cmp['metacells_in_03'] = mc_targets['n_metacells']
    print('Median pseudobulk unit vs the UMI budget one metacell was sized to:')
    display(cmp.sort_values('ratio').round(0))
except FileNotFoundError:
    print(f'No notebook 03 output at {MC_DIR}; skipping the comparison.')

## Part 2 · Leverage, dropout model and the ambient floor
These are fitted on the **single cells**, not the pseudobulk, because the dropout model describes how
a cell's library size turns expression into counts. The pseudobulk units inherit the per-gene rate
and leverage from that fit.

The per-control table below is the thing to actually look at. A negative control detected in every
cell type, or an order of magnitude above its fellows, is expressed in this tissue and must be
removed — otherwise it raises the floor and starts closing reactions that should stay open.

In [ ]:
leverage, cells = gene_dropout_leverage(
    adata, CELLTYPE_COL, species=SPECIES, symbol_col=SYMBOL_COL,
    and_strategy=AND_STRATEGY, or_strategy=OR_STRATEGY,
    split_isozymes=SPLIT_ISOZYMES, reference='global')
genes_df, summary = dropout_diagnostic(cells, leverage, CELLTYPE_COL,
                                       detection_prob=DETECTION_PROB, model=NULL_MODEL)
print(f'{genes_df["model_gene"].nunique()} model genes measured, '
      f'{len(genes_df)} gene x cell-type rows')

floor, per_control = ambient_floor(genes_df, AMBIENT_CONTROL_SYMBOLS, FLOOR_QUANTILE)
missing = sorted(set(AMBIENT_CONTROL_SYMBOLS) - set(genes_df['symbol'].astype(str)))
print(f'\nAmbient floor (q{FLOOR_QUANTILE:.2f} of control rates): {floor:.3e}')
print(f'Controls not measured in this dataset: {missing}')
print('\nPer control -- drop any detected in every cell type, or far above the rest:')
display(per_control)

In [ ]:
# The floor only matters relative to how deep each cell type is: it decides which cell types
# can distinguish "absent" from "not seen". Anything above the line cannot make the call.
pooled = pb.obs.groupby(CELLTYPE_COL, observed=True)['total_umis'].sum()
ceiling = (-np.log(1 - CONFIDENCE) / pooled).rename('ceiling_rate').to_frame()
ceiling['pooled_umis'] = pooled
ceiling['can_call_off'] = ceiling['ceiling_rate'] <= floor
ceiling = ceiling.sort_values('pooled_umis', ascending=False)
print(f'{int(ceiling["can_call_off"].sum())} of {len(ceiling)} cell types pool enough UMIs to call a gene off')
display(ceiling.round(10))

## Part 3 · Gene and reaction classes
`pooled` = all three samples detect it · `partial` = they disagree · `off` = none detect it and the
cell type is deep enough to say so · `uncertain` = none detect it and we cannot tell.

`partial` means something different here than in notebook 03. There it meant "a larger metacell would
see it"; here nothing larger exists, so it means **the replicates disagree** — which is the more
useful warning when the samples are the replicates.

In [ ]:
gene_classes = pseudobulk_gene_classes(
    pb, genes_df, CELLTYPE_COL, SAMPLE_COL,
    control_genes=AMBIENT_CONTROL_SYMBOLS, floor_quantile=FLOOR_QUANTILE, confidence=CONFIDENCE)

with open(resolve_model_path('default'), 'r', encoding='utf-8') as f:
    model_json = json.load(f)
reaction_classes = classify_reactions(model_json, gene_classes, set(cells.var_names.astype(str)),
                                      SPECIES, AND_STRATEGY, OR_STRATEGY)

print('genes:   ', gene_classes['class'].value_counts().to_dict())
print('reactions:', reaction_classes['class'].value_counts().to_dict())
display(pd.crosstab(gene_classes['group'], gene_classes['class']))
display(pd.crosstab(reaction_classes['group'], reaction_classes['class']))

In [ ]:
# Leverage is what matters, not gene counts: a gene the model barely depends on costs little.
lev = (gene_classes.groupby('class')['leverage'].sum() / gene_classes['leverage'].sum()).sort_values()
print('Share of total leverage by class:')
print((lev * 100).round(1).to_string())

# Reactions that get transcriptomic bounds, weighted by cell type
share = (pd.crosstab(reaction_classes['group'], reaction_classes['class'], normalize='index') * 100).round(1)
display(share.sort_values('pooled', ascending=False))

## Part 4 · Does replication agree with the dropout model?
The model predicts how many cells a gene needs before it is detected. Replication says whether all
three samples actually saw it. If the two agree, the model is usable as a per-gene reliability score
on datasets without replicates; if they do not, replication is the one to trust, because it is
evidence rather than prediction.

In [ ]:
g = gene_classes.copy()
g['n_cells'] = g['group'].map(adata.obs.groupby(CELLTYPE_COL, observed=True).size())
agree = (g.groupby('class')
         .agg(rows=('model_gene', 'size'),
              median_units_needed=('units_needed', 'median'),
              median_rate=('rate', 'median'),
              leverage=('leverage', 'sum')).round(4))
display(agree)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
ax = axes[0]
for klass, d in g.groupby('class'):
    finite = d['units_needed'].replace([np.inf, -np.inf], np.nan).dropna()
    if len(finite):
        ax.hist(np.log10(finite.clip(lower=0.1)), bins=40, alpha=0.55, label=klass)
ax.set_xlabel('log10(cells needed for detection, from the model)')
ax.set_ylabel('gene x cell-type rows')
ax.set_title('Does the model separate the replication classes?', fontsize=10)
ax.legend(fontsize=8)

ax = axes[1]
frac = (g.assign(ok=g['n_detected'] / g['n_samples'])
        .groupby(pd.cut(np.log10(g['units_needed'].replace([np.inf], 1e6).clip(lower=0.1)), bins=20),
                 observed=True)['ok'].mean())
ax.plot([i.mid for i in frac.index], frac.values, marker='o', lw=1)
ax.set_xlabel('log10(cells needed for detection)')
ax.set_ylabel('mean share of samples detecting')
ax.set_ylim(-0.05, 1.05)
ax.set_title('Predicted difficulty vs observed reproducibility', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Genes the model calls easy but the samples disagree on -- the interesting failures
suspect = g[(g['class'] == 'partial') & (g['units_needed'] < g['n_cells'] / 10)]
print(f'{len(suspect)} gene x cell-type rows the model expects to be easy, yet the samples disagree:')
display(suspect.nlargest(20, 'leverage')[
    ['group', 'symbol', 'leverage', 'units_needed', 'n_cells', 'n_detected', 'rate']].round(4))

## Part 5 · Compare with the metacell classification
How much does moving to pseudobulk change which reactions get transcriptomic bounds? Both
classifications cover the same reaction × cell type rows, so they can be crossed directly.

In [ ]:
try:
    mc_rc = pd.read_csv(os.path.join(MC_DIR, 'reaction_classes.csv'))
    a = mc_rc.set_index(['group', 'reaction_id'])['class'].rename('metacells')
    b = reaction_classes.set_index(['group', 'reaction_id'])['class'].rename('pseudobulk')
    both = pd.concat([a, b.reindex(a.index)], axis=1).dropna()
    display(pd.crosstab(both['metacells'], both['pseudobulk']))
    moved = int((both['metacells'] != both['pseudobulk']).sum())
    print(f'{moved} of {len(both)} reaction x cell-type rows change class ({moved / len(both):.1%})')
except FileNotFoundError:
    print(f'No notebook 03 reaction classes at {MC_DIR}; skipping the comparison.')

## Save

In [ ]:
pb.write_h5ad(os.path.join(OUT_DIR, 'pseudobulk_E16.h5ad'))
labels.rename('pseudobulk_unit').to_csv(os.path.join(OUT_DIR, 'cell_to_unit.csv'))
gene_classes.to_csv(os.path.join(OUT_DIR, 'gene_classes.csv'), index=False)
reaction_classes.to_csv(os.path.join(OUT_DIR, 'reaction_classes.csv'), index=False)
genes_df.to_csv(os.path.join(OUT_DIR, 'gene_dropout.csv'), index=False)
per_control.to_csv(os.path.join(OUT_DIR, 'ambient_controls.csv'))
ceiling.to_csv(os.path.join(OUT_DIR, 'off_call_capability.csv'))
depth.to_csv(os.path.join(OUT_DIR, 'unit_depth.csv'))
with open(os.path.join(OUT_DIR, 'settings.json'), 'w') as f:
    json.dump({'and_strategy': AND_STRATEGY, 'or_strategy': OR_STRATEGY,
               'split_isozymes': SPLIT_ISOZYMES, 'detection_prob': DETECTION_PROB,
               'null_model': NULL_MODEL, 'confidence': CONFIDENCE,
               'floor_quantile': FLOOR_QUANTILE, 'ambient_floor': float(floor),
               'ambient_controls': list(AMBIENT_CONTROL_SYMBOLS),
               'controls_not_measured': missing}, f, indent=2)
print('Saved to', OUT_DIR)

## Sending results back
1. **File → Save Notebook As…** → `metabolic/results/04_pseudobulk_E16/04_pseudobulk_E16_run.ipynb`
2. On the HPC, zip the results directory:
```bash
cd /scratch/prj/crb_inner_ear/k2147692/metabolic/results && rm -f 04_results.zip && zip -rq 04_results.zip 04_pseudobulk_E16
```
3. Then from PowerShell on the laptop:
```powershell
Set-Location "$env:USERPROFILE\OneDrive - King's College London\Documents\Metabolic-results"
scp -o MACs=hmac-sha2-512 k2147692@hpc.create.kcl.ac.uk:/scratch/prj/crb_inner_ear/k2147692/metabolic/results/04_results.zip .
Expand-Archive -Path .\04_results.zip -DestinationPath . -Force
```
4. Leave the repo clean for the next pull:
```bash
cd /scratch/prj/crb_inner_ear/k2147692/metabolic/code/Metabolic-pipeline && git checkout -- notebooks/
```